# 🌳 RAPTOR — Search at Multiple Levels of Detail

## The Problem with Flat RAG

In basic RAG, all chunks are at the same level of detail.

```
Flat RAG:
  Chunk 1: "Alice approved PostgreSQL in the Q3 review."
  Chunk 2: "Bob wrote the database design doc."
  Chunk 3: "The Q3 review covered authentication and storage."
  ...
  All chunks are equal — no big-picture view
```

If someone asks a high-level question like **"What was decided in Q3?"**, flat RAG misses the forest for the trees.

## The RAPTOR Idea

Build a **tree** of summaries. Search at any level.

```
Level 2 (root):  "Q3 review finalised tech stack: PostgreSQL + React."
                          ↑ summary of...
Level 1:         "Database: PostgreSQL"    "Frontend: React"
                      ↑ summary of...           ↑ summary of...
Level 0 (leaves): [original chunks]          [original chunks]
```

- **High-level question** → search high in the tree
- **Detailed question** → search low in the tree
- **Both** → RAPTOR searches all levels

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

Ready!


In [2]:
# Level 0 — raw leaf chunks
leaf_chunks = [
    "Alice approved using PostgreSQL for the primary database.",
    "Bob proposed PostgreSQL because of its JSONB support and reliability.",
    "The team decided React would be used for the frontend UI.",
    "Carol pushed for React citing its large ecosystem and team familiarity.",
    "Authentication will use JWT tokens with a 24-hour expiry.",
    "Dave raised concerns about JWT and suggested session-based auth instead.",
]

# Level 1 — summaries of related leaf chunks
# (In production these are generated by an LLM)
level1_summaries = [
    "The database decision: PostgreSQL was selected as the primary database, approved by Alice after Bob's proposal citing JSONB support.",
    "The frontend decision: React was chosen for the UI, championed by Carol due to the team's existing experience.",
    "The authentication decision: JWT with 24-hour expiry was chosen, despite Dave's concerns about session-based alternatives.",
]

# Level 2 — root summary
level2_summary = [
    "Q3 architecture review outcome: The team finalised PostgreSQL for the database, React for the frontend, and JWT for authentication."
]

# Combine all levels with metadata
all_nodes = []
for text in leaf_chunks:      all_nodes.append({"level": 0, "text": text})
for text in level1_summaries: all_nodes.append({"level": 1, "text": text})
for text in level2_summary:   all_nodes.append({"level": 2, "text": text})

all_texts  = [n["text"] for n in all_nodes]
embeddings = model.encode(all_texts)

print(f"Tree built: {len(leaf_chunks)} leaves, {len(level1_summaries)} L1 summaries, {len(level2_summary)} root")

Tree built: 6 leaves, 3 L1 summaries, 1 root


In [3]:
# RAPTOR search — searches ALL levels, returns results labelled by level

def raptor_search(question, top_k=3):
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    level_labels = {0: "Detail", 1: "Summary", 2: "Overview"}

    print(f"Query: '{question}'")
    print("-" * 60)
    for rank, idx in enumerate(top_idx, 1):
        node  = all_nodes[idx]
        label = level_labels[node["level"]]
        print(f"  {rank}. [{scores[idx]:.3f}] [{label}] {node['text']}")
    print()

# High-level question → should hit overview/summary nodes
raptor_search("What was decided in the Q3 review?")

# Detailed question → should hit leaf nodes
raptor_search("What were the concerns about authentication?")

Query: 'What was decided in the Q3 review?'
------------------------------------------------------------
  1. [0.517] [Overview] Q3 architecture review outcome: The team finalised PostgreSQL for the database, React for the frontend, and JWT for authentication.
  2. [0.351] [Summary] The frontend decision: React was chosen for the UI, championed by Carol due to the team's existing experience.
  3. [0.278] [Summary] The database decision: PostgreSQL was selected as the primary database, approved by Alice after Bob's proposal citing JSONB support.

Query: 'What were the concerns about authentication?'
------------------------------------------------------------
  1. [0.462] [Summary] The authentication decision: JWT with 24-hour expiry was chosen, despite Dave's concerns about session-based alternatives.
  2. [0.429] [Detail] Dave raised concerns about JWT and suggested session-based auth instead.
  3. [0.353] [Overview] Q3 architecture review outcome: The team finalised PostgreSQL for th

In [4]:
# See how flat search vs RAPTOR differs on high-level questions

def flat_search(question, top_k=3):
    """Only searches leaf nodes (like basic RAG)"""
    leaf_indices = [i for i, n in enumerate(all_nodes) if n["level"] == 0]
    leaf_embs    = embeddings[leaf_indices]
    q_emb        = model.encode(question)
    scores       = cosine_similarity([q_emb], leaf_embs)[0]
    top_local    = np.argsort(scores)[::-1][:top_k]
    return [all_nodes[leaf_indices[i]]["text"] for i in top_local]

question = "What was the overall outcome of the Q3 review?"

print("Flat search (leaves only):")
for r in flat_search(question):
    print(f"  → {r[:70]}")

print("\nRAPTOR search (all levels):")
raptor_search(question, top_k=3)

Flat search (leaves only):
  → Dave raised concerns about JWT and suggested session-based auth instea
  → The team decided React would be used for the frontend UI.
  → Carol pushed for React citing its large ecosystem and team familiarity

RAPTOR search (all levels):
Query: 'What was the overall outcome of the Q3 review?'
------------------------------------------------------------
  1. [0.483] [Overview] Q3 architecture review outcome: The team finalised PostgreSQL for the database, React for the frontend, and JWT for authentication.
  2. [0.260] [Summary] The frontend decision: React was chosen for the UI, championed by Carol due to the team's existing experience.
  3. [0.238] [Detail] Dave raised concerns about JWT and suggested session-based auth instead.



## Key Takeaways

| | Flat RAG | RAPTOR |
|---|---|---|
| **High-level questions** | ❌ Misses big picture | ✅ Hits summary nodes |
| **Detail questions** | ✅ Good | ✅ Good |
| **Setup complexity** | Simple | Requires LLM to generate summaries |
| **Best for** | Short docs | Long docs, books, large corpora |

**Paper:** RAPTOR: Recursive Abstractive Processing for Tree-Organized Retrieval (Sarthi et al., 2024)